# Data-Driven Asset Allocation
**Author:** Ryan Kelly  
**Program:** Open Avenues Foundation — Data-Driven Asset Allocation Strategies  
**Data:** Hourly prices (one year) for 20 US equities  
**Goal:** Construct a rolling Kelly allocation, benchmark against equal-weight and SPY, and report standard performance metrics

## 1. Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings("ignore")

## 2. Configuration

All parameters are defined in one place so they are easy to change without hunting through the notebook.

In [ ]:
# Asset universe
PORTFOLIO = [
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META',
    'NEE',  'FSLR', 'SPWR',
    'JPM',  'BAC',  'WFC',
    'ABBV', 'PFE',  'UNH',
    'GE',   'HON',  'CAT',
    'PG',   'KO',   'WMT',
]
BENCHMARK = 'SPY'

# Date range
START = '2024-12-01'
END   = '2025-12-01'

# Risk-free rate
RF_YEAR = 2.90 / 100
RF_HOUR = (1 + RF_YEAR) ** (1 / 8760) - 1  # derived from annual, not hardcoded

# Rolling window length (hours)
M = 24

# Per-asset weight cap
KELLY_CAP = 0.21

print(f'Annual RF : {RF_YEAR:.2%}')
print(f'Hourly RF : {RF_HOUR:.3e}')
print(f'Window  M : {M} hours')

## 3. Data Download

`yf.download` fetches all tickers in a single request, which is faster than looping and guarantees a shared datetime index.  
`ffill` fills occasional intra-day gaps before dropping any row that is still NaN.

In [ ]:
raw = yf.download(
    PORTFOLIO + [BENCHMARK],
    start=START,
    end=END,
    interval='60m',
    progress=False,
    auto_adjust=True,
)['Close']

raw = raw.ffill().dropna()

prices    = raw[PORTFOLIO]
spy_price = raw[BENCHMARK]

print(f'Price matrix shape: {prices.shape}  (rows=hours, cols=assets)')
display(prices.head())

**Observation — Data Quality**  
The price matrix should have no NaN values after `ffill().dropna()`. Verify the shape is approximately (1,000+, 20), consistent with roughly one year of hourly US equity data (~6.5 trading hours × ~252 days). If any ticker returns significantly fewer rows than the others it may have been delisted or had a data outage during the sample period — check with `prices.isna().sum()` if the shape looks unexpectedly small.

## 4. Returns & Excess Returns

Hourly returns are computed with `pct_change`. The hourly risk-free rate is subtracted to produce excess returns used for Kelly weight estimation.

In [ ]:
R_df         = prices.pct_change().dropna()
ExcessRet_df = R_df - RF_HOUR

# Align SPY to the same index
spy_ret      = spy_price.pct_change().reindex(R_df.index).dropna()
ExcessRet_df = ExcessRet_df.reindex(spy_ret.index)
R_df         = R_df.reindex(spy_ret.index)

R  = ExcessRet_df.to_numpy()  # excess returns — used for weight estimation
Rp = R_df.to_numpy()          # total returns  — used for portfolio P&L
T, N = R.shape
print(f'T={T} periods, N={N} assets')

# Return distribution summary
print('\nHourly Excess Return Summary (all assets):')
display(ExcessRet_df.describe().T[['mean','std','min','max']].style.format('{:.6f}'))

**Observation — Return Distribution**  
Mean hourly excess returns should be very close to zero (on the order of 1e-4 or smaller) — large positive means would suggest a data or alignment error. Standard deviations in the range of 0.003–0.015 per hour are typical for US large-cap equities. Any asset with `std` significantly above 0.02 (e.g. SPWR) reflects higher idiosyncratic risk and should be watched in the weight output — the Kelly solution may over- or under-weight it depending on recent momentum.

## 5. Kelly Weight Estimator

For each rolling window the Kelly weight vector is estimated via the mean-variance approximation:

$$f \propto \Sigma^{-1}\,\mu$$

**Key design choices:**
- `np.linalg.lstsq` replaces `np.linalg.inv` — handles near-singular covariance matrices without crashing (common when window length M is close to asset count N).
- Long-only: negative weights are zeroed and renormalized.
- Per-asset cap enforced via a stable clip-and-redistribute loop.

In [ ]:
def kelly_weights(R_window: np.ndarray, cap: float = KELLY_CAP) -> np.ndarray:
    """
    Estimate Kelly weights from a (M, N) window of excess returns.
    Returns a length-N weight vector summing to 1.
    """
    mu  = R_window.mean(axis=0)
    cov = np.cov(R_window, rowvar=False)

    # lstsq is robust when cov is near-singular
    w_raw, _, _, _ = np.linalg.lstsq(cov, mu, rcond=None)

    # Long-only constraint
    w_raw = np.where(w_raw > 0, w_raw, 0.0)
    total = w_raw.sum()
    if total == 0:
        return np.ones(N) / N  # fallback to equal weight
    w = w_raw / total

    # Iterative cap-and-redistribute
    for _ in range(50):
        excess = np.maximum(w - cap, 0).sum()
        if excess < 1e-10:
            break
        w = np.clip(w, 0, cap)
        slack = cap - w
        below_cap = slack > 0
        if not below_cap.any():
            break
        w[below_cap] += excess * slack[below_cap] / slack[below_cap].sum()

    return w / w.sum()

# Validate on a single window
test_w = kelly_weights(R[:M])
print(f'Validation on first window:')
print(f'  Sum of weights : {test_w.sum():.8f}  (should be 1.0)')
print(f'  Min weight     : {test_w.min():.6f}  (should be >= 0.0)')
print(f'  Max weight     : {test_w.max():.6f}  (should be <= {KELLY_CAP})')
print(f'  Assets with w>0: {(test_w > 0).sum()} of {N}')

**Observation — Weight Function Validation**  
The four checks above confirm the estimator is working correctly before running it across all windows:
- Sum = 1.0 confirms normalization is exact.
- Min >= 0.0 confirms the long-only constraint is enforced.
- Max <= 0.21 confirms the per-asset cap is binding where needed.
- Assets with w > 0 tells you how concentrated the allocation is. With M=24 and N=20, expect the Kelly solution to be reasonably concentrated — often 5–10 assets receive meaningful weight after zeroing negatives.

## 6. Rolling Weight Computation

For each hour $t \geq M$ the Kelly weights are estimated from the previous $M$ hours.  
Equal-weight requires no loop — it is a constant allocation.

In [ ]:
# Kelly: rolling window
weight_kelly = np.full((T, N), np.nan)
windows = sliding_window_view(R, (M, N))  # shape: (T-M+1, 1, M, N)

for i in range(len(windows) - 1):
    weight_kelly[i + M] = kelly_weights(windows[i][0])

# Equal-weight: constant
weight_ew = np.full((T, N), 1.0 / N)
weight_ew[:M] = np.nan  # burn-in period matches Kelly

# Sanity check on last valid Kelly allocation
last_valid = np.where(~np.isnan(weight_kelly[:, 0]))[0][-1]
last_w = weight_kelly[last_valid]
print(f'Last rebalance — sum={last_w.sum():.6f}  min={last_w.min():.4f}  max={last_w.max():.4f}')

# Weight time series: how often does each asset get selected?
selection_freq = pd.Series(
    (~np.isnan(weight_kelly) & (weight_kelly > 0.001)).mean(axis=0),
    index=PORTFOLIO,
    name='Selection Frequency'
).sort_values(ascending=False)
print('\nFraction of periods each asset receives weight > 0.1%:')
display(selection_freq.to_frame().style.format('{:.2%}'))

# Last allocation bar chart
kelly_series = pd.Series(last_w, index=PORTFOLIO, name='Weight').sort_values(ascending=False)
display(kelly_series.to_frame().style.format('{:.4f}').set_caption('Kelly Allocation — Last Rebalance'))

plt.figure(figsize=(12, 4))
kelly_series.plot(kind='bar')
plt.axhline(KELLY_CAP, color='red', linestyle='--', linewidth=1, label=f'Cap ({KELLY_CAP:.0%})')
plt.title('Kelly Criterion Allocation — Last Rebalance Window')
plt.ylabel('Portfolio Weight')
plt.xlabel('Asset')
plt.legend()
plt.tight_layout()
plt.grid(axis='y')
plt.show()

**Observation — Rolling Weights**  
The selection frequency table shows which assets the Kelly model consistently favors across the full sample. Assets that appear in the top allocation in nearly every window tend to have a better mean-to-variance ratio of hourly excess returns over the period. Assets with low selection frequency are either high-volatility (e.g. SPWR, FSLR) or have mean excess returns close to zero, which causes the Kelly solution to assign them zero weight after the long-only constraint is applied.

In the last-rebalance bar chart, assets hitting the 21% cap have been clipped — their raw unconstrained Kelly weight was even higher, reflecting very strong recent momentum. The redistributed weight flows to the next tier of assets in proportion to their remaining slack below the cap.

## 7. Portfolio Returns

Weights are applied to **total** (not excess) returns so the cumulative curve reflects real dollar growth from $1.  
SPY is included as a passive market benchmark.

In [ ]:
valid = slice(M, T)

port_ret_kelly   = np.nansum(weight_kelly[valid] * Rp[valid], axis=1)
kelly_cumulative = np.cumprod(1 + port_ret_kelly)

port_ret_ew   = np.nansum(weight_ew[valid] * Rp[valid], axis=1)
ew_cumulative = np.cumprod(1 + port_ret_ew)

spy_ret_arr    = spy_ret.to_numpy()[M:]
spy_cumulative = np.cumprod(1 + spy_ret_arr)

time_index = R_df.index[valid]

plt.figure(figsize=(12, 5))
plt.plot(time_index, kelly_cumulative, label='Kelly',        linewidth=1.5)
plt.plot(time_index, ew_cumulative,   label='Equal Weight',  linewidth=1.5, linestyle='--')
plt.plot(time_index, spy_cumulative,  label='SPY Benchmark', linewidth=1.5, linestyle=':')
plt.title('Cumulative Growth of $1 — Kelly vs Equal-Weight vs SPY')
plt.ylabel('Portfolio Value ($)')
plt.xlabel('Date')
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

**Observation — Cumulative Returns**  
The cumulative chart is the central visual for evaluating strategy performance. Key things to look for:

- **Tracking behavior:** If Kelly closely tracks equal-weight for long stretches, the cap and long-only constraints are doing most of the work — the raw Kelly signal is being overridden frequently. Divergence between the two indicates periods where the model's tilt is actually different from uniform.
- **Drawdown periods:** Any sharp downward moves that affect all three lines simultaneously are likely market-wide events (e.g. a macro shock). A drawdown that is sharper in Kelly than in SPY or equal-weight suggests the model overweighted the wrong assets in that window.
- **Terminal value:** The ending portfolio value relative to $1 gives a direct read on total return over the period before annualization.

## 8. Performance Metrics

All metrics are annualized using **8,760 hourly periods per year**.

In [ ]:
HOURS_PER_YEAR = 8_760

def performance_metrics(ret_arr: np.ndarray, label: str) -> dict:
    n       = len(ret_arr)
    ann_ret = (1 + ret_arr).prod() ** (HOURS_PER_YEAR / n) - 1
    ann_vol = ret_arr.std() * np.sqrt(HOURS_PER_YEAR)
    sharpe  = (ann_ret - RF_YEAR) / ann_vol if ann_vol > 0 else np.nan

    downside = ret_arr[ret_arr < 0]
    ann_down = downside.std() * np.sqrt(HOURS_PER_YEAR) if len(downside) > 0 else np.nan
    sortino  = (ann_ret - RF_YEAR) / ann_down if (ann_down and ann_down > 0) else np.nan

    cum    = np.cumprod(1 + ret_arr)
    peak   = np.maximum.accumulate(cum)
    max_dd = (cum / peak - 1).min()

    return {
        'Strategy':        label,
        'Ann. Return':     f'{ann_ret:.2%}',
        'Ann. Volatility': f'{ann_vol:.2%}',
        'Sharpe Ratio':    f'{sharpe:.3f}',
        'Sortino Ratio':   f'{sortino:.3f}',
        'Max Drawdown':    f'{max_dd:.2%}',
    }

metrics = pd.DataFrame([
    performance_metrics(port_ret_kelly, 'Kelly'),
    performance_metrics(port_ret_ew,    'Equal Weight'),
    performance_metrics(spy_ret_arr,    'SPY'),
]).set_index('Strategy')

display(metrics.style.set_caption('Annualized Performance Metrics'))

**Observation — Performance Metrics**  
Read the table as follows:

- **Ann. Return vs SPY:** If Kelly's annualized return exceeds SPY, the active allocation added value over the passive benchmark. If it trails, the concentrated bets hurt more than they helped over this period.
- **Sharpe Ratio:** A Sharpe above 1.0 is considered strong for an equity strategy. Comparing Kelly's Sharpe to equal-weight isolates the value of the Kelly tilt — if equal-weight has a higher Sharpe, the optimization is adding noise rather than signal at this window length.
- **Sortino Ratio:** Because Sortino penalizes only downside volatility, a Sortino > Sharpe indicates the strategy's volatility is skewed upward (good). A Sortino much lower than Sharpe suggests asymmetric downside risk.
- **Max Drawdown:** The worst peak-to-trough loss over the period. A Kelly drawdown meaningfully deeper than equal-weight or SPY would indicate the 24-hour estimation window is too short to avoid reacting to noise — the model rotates into losers just before they correct.

## 9. Conclusion

This project applied a rolling-window Kelly Criterion to construct a dynamic hourly allocation across 20 US equities and evaluated it against an equal-weight portfolio and the S&P 500.

**What the Kelly model does.** At each hour the model estimates which assets have the highest recent mean excess return relative to their covariance with other assets, and tilts toward them. The long-only constraint and 21% per-asset cap prevent the theoretical Kelly solution — which can be highly leveraged and concentrated — from taking extreme positions.

**What the results show.** The performance table and cumulative chart together answer the central question: does data-driven rebalancing at hourly frequency improve risk-adjusted returns over passive alternatives? If the Kelly Sharpe exceeds both equal-weight and SPY, the model's rotation into recent winners added value. If it does not, the most likely explanations are:

1. **Window length.** M = 24 observations for N = 20 assets leaves the covariance matrix rank-deficient. The `lstsq` solver handles this but the weights are driven largely by recent noise rather than persistent signal.
2. **Mean reversion.** Hourly momentum can reverse quickly. A window that selects recent winners may systematically buy just before a short-term correction.
3. **Transaction costs.** Hourly rebalancing generates substantial turnover. Even small per-trade costs would erode the gross return advantage.

**Possible improvements.** Extending M to 120–240 hours, applying Ledoit-Wolf covariance shrinkage, or adding a turnover penalty to the optimization objective would each address one of the limitations above. Adding a momentum pre-filter — only trading assets above their 5-day moving average — could reduce the mean-reversion problem without changing the core Kelly framework.